In [ ]:
"""
 [홍석환]
 공백·빈값, ID와 날짜 표기를 표준화하고 동일 내용 중복을 제거합니다. 
 접수 연결과 진료 시간 순서를 검증하고, 서비스 상세합 및 결제·환불의 독립 집계 근거가 일치할 때만 청구액·환불액·순액을 복구합니다. 
 복구 이력을 기록하고 검증을 통과한 행과 검토 대상을 분리하여 정제본과 DQ 자료를 저장합니다.
 """

# encounters: 명세서·ERD 기반 정제 및 교차검증

완료 진료 한 건을 나타내는 테이블입니다. `hospital_명세서.xlsx`의 12·18·20 시트와 `hospital_erd_guide.html`의 접수·실제 진료 / 서비스·진료비·결제 관계를 기준으로 재정제합니다.

원본 수정·데이터 파일 저장·DB 적재 없이 DataFrame으로 결과를 확인합니다. 모든 코드는 이 노트북에 있습니다. .venv 커널로 순서대로 실행하세요.

기존 결과와 달리 접수 ID UNIQUE, 엄격한 시간 순서, 슬래시 날짜 및 하이픈 ID 복구, 참조 데이터의 동일 내용 중복 제거, 독립된 서비스·결제 집계를 반영합니다. 참고 데이터 자체가 오염돼 있으면 교차검증 미완료로 분리합니다.

## 1. 데이터 구조: (행, 열) 개수

In [1]:
from pathlib import Path
import csv, sys, re, zipfile, xml.etree.ElementTree as ET
import polars as pl
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/input/encounters').is_dir())
INPUT=ROOT/'data/input'
COLS=['encounter_id','checkin_id','started_at','ended_at','gross_charge','authorized_refund','net_charge','source_system']
MONEY=['gross_charge','authorized_refund','net_charge']
FILES=sorted((INPUT/'encounters').glob('*.csv'))
assert FILES
for p in FILES:
    with p.open(encoding='utf-8-sig',newline='') as f: assert next(csv.reader(f))==COLS,p.name
raw=pl.concat([pl.scan_csv(p,infer_schema=False,row_index_name='source_record').with_columns(
    pl.lit(p.name).alias('source_file'),(pl.col('source_record')+1).alias('source_record')) for p in FILES]).collect()
print('Python:',sys.executable,'Polars:',pl.__version__)
print('Files:',len(FILES),'Original shape:',(raw.height,len(COLS)))
display(raw.group_by('source_file').len().sort('source_file'))

Python: C:\workspace\medallion_venv\.venv\Scripts\python.exe Polars: 2.0.0
Files: 36 Original shape: (1004999, 8)
shape: (36, 2)
┌────────────────────────┬───────┐
│ source_file            ┆ len   │
│ ---                    ┆ ---   │
│ str                    ┆ u32   │
╞════════════════════════╪═══════╡
│ encounters_2023-01.csv ┆ 28474 │
│ encounters_2023-02.csv ┆ 25125 │
│ encounters_2023-03.csv ┆ 30152 │
│ encounters_2023-04.csv ┆ 25123 │
│ encounters_2023-05.csv ┆ 30159 │
│ …                      ┆ …     │
│ encounters_2025-08.csv ┆ 28472 │
│ encounters_2025-09.csv ┆ 26801 │
│ encounters_2025-10.csv ┆ 30146 │
│ encounters_2025-11.csv ┆ 25122 │
│ encounters_2025-12.csv ┆ 30156 │
└────────────────────────┴───────┘


## 2. 데이터 내 컬럼 세부 정보
명세서상 모든 컬럼은 NOT NULL입니다. 날짜는 Asia/Seoul의 로컬 시각이며 MySQL DATETIME과 대응하도록 시간대 없는 로컬 값으로 유지합니다. 금액은 INT 범위를 검사합니다.

In [2]:
details=pl.DataFrame({
    'column':COLS,
    'description':['진료 ID','접수 ID','진료 시작 시각','진료 종료 시각','총 청구액(KRW)','승인 환불액(KRW)','최종 청구액(KRW)','원천 시스템'],
    'mysql_dtype':['VARCHAR(32)','VARCHAR(32)','DATETIME','DATETIME','INT','INT','INT','VARCHAR(8)'],
    'constraint':['PK, ENC_9자리 숫자','FK+UNIQUE, CHK_9자리 숫자','접수 이후, 종료 이전','시작 이후, 세션 종료 이하','서비스 상세 합계','0 이상 총액 이하','총액 - 환불, 결제 - 환불','LEGACY_A / LEGACY_B'],
    'raw_null_count':[raw[c].null_count() for c in COLS],
    'raw_unique_count':[raw[c].n_unique() for c in COLS]})
display(details)
# 참조 문서의 관련 시트와 ERD 텍스트도 노트북에서 직접 확인
ns={'m':'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
with zipfile.ZipFile(INPUT/'hospital_명세서.xlsx') as z:
    strings=[''.join(e.itertext()) for e in ET.fromstring(z.read('xl/sharedStrings.xml')).findall('m:si',ns)]
    rels={r.attrib['Id']:r.attrib['Target'] for r in ET.fromstring(z.read('xl/_rels/workbook.xml.rels'))}
    for sh in ET.fromstring(z.read('xl/workbook.xml')).find('m:sheets',ns):
        if not sh.attrib['name'].startswith(('12_','18_','20_')):continue
        t=rels[sh.attrib['{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id']].lstrip('/')
        t=t if t.startswith('xl/') else 'xl/'+t
        print(sh.attrib['name'])
        for row in ET.fromstring(z.read(t)).findall('.//m:row',ns):
            vals=[]
            for c in row:
                v=c.find('m:v',ns);v=v.text if v is not None else ''
                v=strings[int(v)] if c.attrib.get('t')=='s' and v else v
                if v and (not vals or vals[-1]!=v):vals.append(v)
            text=' | '.join(vals)
            if re.search('encounters|B08|B09|B10|B11|B12|B13|B15|gross_charge|net_charge|authorized_refund|started_at|ended_at|uq_enc',text):print(text)
html=(INPUT/'hospital_erd_guide.html').read_text(encoding='utf-8')
erd_text=[s.strip() for s in re.findall(r'<!--(.*?)-->',html,re.S)]
print([s for s in erd_text if ('청구액' in s or '서비스 상세 금액' in s or 'FK UK' in s)])

shape: (8, 6)
┌────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬────────────────┐
│ column         ┆ description    ┆ mysql_dtype ┆ constraint     ┆ raw_null_count ┆ raw_unique_cou │
│ ---            ┆ ---            ┆ ---         ┆ ---            ┆ ---            ┆ nt             │
│ str            ┆ str            ┆ str         ┆ str            ┆ i64            ┆ ---            │
│                ┆                ┆             ┆                ┆                ┆ i64            │
╞════════════════╪════════════════╪═════════════╪════════════════╪════════════════╪════════════════╡
│ encounter_id   ┆ 진료 ID        ┆ VARCHAR(32) ┆ PK, ENC_9자리  ┆ 0              ┆ 1000000        │
│                ┆                ┆             ┆ 숫자           ┆                ┆                │
│ checkin_id     ┆ 접수 ID        ┆ VARCHAR(32) ┆ FK+UNIQUE,     ┆ 10004          ┆ 979998         │
│                ┆                ┆             ┆ CHK_9자리 숫자 ┆                ┆      

## 3. 데이터 행(레코드) 샘플

In [3]:
display(raw.head(5))
display(raw.tail(5))
display(raw.sample(n=10,seed=42))

shape: (5, 10)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ source_re ┆ encounter ┆ checkin_i ┆ started_a ┆ … ┆ authorize ┆ net_charg ┆ source_sy ┆ source_f │
│ cord      ┆ _id       ┆ d         ┆ t         ┆   ┆ d_refund  ┆ e         ┆ stem      ┆ ile      │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ u32       ┆ str       ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 1         ┆ ENC_00000 ┆ CHK_00000 ┆ 2023-01-0 ┆ … ┆ 0         ┆ 21000     ┆ LEGACY_A  ┆ encounte │
│           ┆ 0001      ┆ 0001      ┆ 2         ┆   ┆           ┆           ┆           ┆ rs_2023- │
│           ┆           ┆           ┆ 07:59:00  ┆   ┆           ┆           ┆           ┆ 01.csv   │
│ 2         ┆ ENC_00000 ┆ CHK_00000 ┆ 2023-01-3 ┆ … ┆ 0         ┆ 35000     

## 4. 클리닝 전 원본(오염된) 상태
원본 기준의 누락·중복·날짜·금액 문제를 기록합니다. 명세서에서 허용한 표기 차이와 복구할 수 없는 값을 구분합니다.

In [4]:
raw_rules={}
for c in COLS:
    raw_rules['missing_'+c]=pl.col(c).is_null()|(pl.col(c).str.strip_chars()=='')
raw_rules['duplicate_content']=pl.struct(COLS).is_duplicated()
raw_rules['duplicate_encounter_id']=pl.col('encounter_id').is_duplicated()
raw_rules['slash_started_at']=pl.col('started_at').str.contains('/')
for c in ['started_at','ended_at']:
    s=pl.col(c).str.strip_chars()
    raw_rules['invalid_datetime_'+c]=s.is_not_null()&(s!='')&s.str.to_datetime('%Y-%m-%d %H:%M:%S',strict=False).is_null()
for c in MONEY:
    raw_rules['invalid_integer_'+c]=pl.col(c).is_not_null()&pl.col(c).cast(pl.Int64,strict=False).is_null()
a={c:pl.col(c).cast(pl.Int64,strict=False) for c in MONEY}
raw_rules['negative_amount']=pl.any_horizontal([v<0 for v in a.values()])
raw_rules['net_mismatch']=a['net_charge']!=a['gross_charge']-a['authorized_refund']
raw_flags=raw.select([e.fill_null(False).alias(k) for k,e in raw_rules.items()])
raw_report=pl.DataFrame({'rule':list(raw_rules),'rows':[raw_flags[k].sum() for k in raw_rules]}).sort('rows',descending=True)
display(raw_report)
for k in ['invalid_datetime_started_at','missing_checkin_id','negative_amount','net_mismatch']:
    print(k);display(raw.filter(raw_flags[k]).head(3))

shape: (18, 2)
┌─────────────────────────────────┬───────┐
│ rule                            ┆ rows  │
│ ---                             ┆ ---   │
│ str                             ┆ i64   │
╞═════════════════════════════════╪═══════╡
│ invalid_datetime_started_at     ┆ 19996 │
│ missing_checkin_id              ┆ 10004 │
│ negative_amount                 ┆ 10004 │
│ net_mismatch                    ┆ 10004 │
│ slash_started_at                ┆ 10000 │
│ …                               ┆ …     │
│ missing_source_system           ┆ 0     │
│ invalid_datetime_ended_at       ┆ 0     │
│ invalid_integer_gross_charge    ┆ 0     │
│ invalid_integer_authorized_ref… ┆ 0     │
│ invalid_integer_net_charge      ┆ 0     │
└─────────────────────────────────┴───────┘
invalid_datetime_started_at
shape: (3, 10)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ source_re ┆ encounter ┆ checkin_i ┆ started_a ┆ … ┆ authorize ┆ net_charg ┆ source_sy ┆ sou

## 5. 클리닝 로직/논리

**문서 적용 기준**
- B08: 진료는 SEEN 접수와 1:1 연결. LEFT/REJECTED 접수에 진료가 있으면 오류입니다.
- B09: arrival_at ≤ checked_in_at ≤ started_at < ended_at ≤ session_end. 시작=종료도 오류입니다.
- B10~B13: 서비스 버전·단가·상세합과 결제·환불을 별도로 검증합니다. 여러 상세·여러 결제를 직접 조인해 중복 합산하지 않습니다.
- B15: 공백 제거, ID 하이픈→밑줄, 슬래시 날짜→하이픈 형식은 결정적인 표준화입니다. 없는 날짜나 연결 ID를 숫자 패턴으로 추측하지 않습니다.

**복구 정책**
- 교차검증에 사용할 참조 데이터도 동일 내용 중복을 제거하고 키 충돌·자료형·규칙을 검증합니다. 원본 참조 테이블을 수정하지 않습니다.
- 서비스 총액과 결제 총액이 일치하고 양쪽 근거에 오류가 없을 때만 총액의 대체 근거로 사용합니다.
- 환불액은 유효한 원결제에 연결된 환불합과 기존 총액−순액이 일치할 때만 복구합니다.
- 순액은 총액−승인환불과 결제합−환불합이 일치할 때만 복구합니다.
- 승인환불과 실제환불의 차이는 실행 시점 차이일 수 있으므로 임의로 맞추지 않습니다.
- 복구 변경사항은 원본 행·컬럼·전후값·근거로 기록합니다. 추측이 필요한 ID, 불가능한 날짜, 상충된 근거는 검토 대상으로 보존합니다.
- 참조 검증 미완료는 진료 자체 오류와 별도의 검토 사유로 기록합니다. SEEN 접수의 진료 누락도 별도 표로 제공합니다.

In [5]:
def strip_text(df,columns):
    return df.with_columns([pl.when(pl.col(c).str.strip_chars()=='').then(None).otherwise(pl.col(c).str.strip_chars()).alias(c) for c in columns])
def canonical_ids(df,columns):
    return df.with_columns([pl.col(c).str.replace_all('-','_').alias(c) for c in columns])
def parse_dt(c):
    return pl.col(c).str.replace_all('/','-').str.to_datetime('%Y-%m-%d %H:%M:%S',strict=False)
def load(table,ids):
    f=sorted((INPUT/table).glob('*.csv'));assert f,table
    d=pl.scan_csv(f,infer_schema=False).collect()
    d=canonical_ids(strip_text(d,d.columns),ids)
    return d.unique(maintain_order=True)
def unique_reference(df,key):
    counts=df.filter(pl.col(key).is_not_null()).group_by(key).len().rename({'len':'reference_count'})
    unique=df.join(counts.filter(pl.col('reference_count')==1),on=key,validate='m:1').drop('reference_count')
    return counts,unique
mappings=load('legacy_code_mappings',['mapping_id'])
def map_codes(df,column,domain):
    m=mappings.filter(pl.col('domain')==domain).select('source_system',pl.col('raw_code').alias(column),'standard_code').unique()
    # 충돌 매핑은 사용하지 않음
    m=m.filter(pl.len().over(['source_system',column])==1)
    return df.join(m,on=['source_system',column],how='left',validate='m:1').with_columns(pl.coalesce('standard_code',column).alias(column)).drop('standard_code')
text=raw.with_columns([pl.col(c).alias('raw_'+c) for c in COLS])
text=canonical_ids(strip_text(text,COLS),['encounter_id','checkin_id']).with_columns([
    pl.col(c).str.replace_all('/','-').alias(c) for c in ['started_at','ended_at']])
duplicate_rows=text.filter(pl.struct(COLS).is_duplicated())
base=text.unique(subset=COLS,keep='first',maintain_order=True).with_columns([
    parse_dt('started_at'),parse_dt('ended_at'),*[pl.col(c).cast(pl.Int64,strict=False) for c in MONEY]])
removed_duplicate_count=raw.height-base.height

In [6]:
# 접수와 세션: 동일 내용 중복만 통합, 충돌 ID는 참조 제외
checkins=map_codes(load('checkins',['checkin_id','patient_id','session_id','appointment_id']),'outcome','outcome')
checkins=checkins.unique().with_columns(parse_dt('arrival_at'),parse_dt('checked_in_at'))
chk_counts,chk=unique_reference(checkins,'checkin_id')
sessions=load('clinic_sessions',['session_id','hospital_department_id','assignment_id']).with_columns(parse_dt('session_end'))
ses_counts,ses=unique_reference(sessions,'session_id')
chk=chk.join(ses.select('session_id','session_end'),on='session_id',how='left',validate='m:1')
chk=chk.select('checkin_id','arrival_at','checked_in_at','session_end','outcome',pl.col('source_system').alias('checkin_source_system'))
base=base.join(chk_counts.rename({'reference_count':'checkin_reference_count'}),on='checkin_id',how='left',validate='m:1').join(chk,on='checkin_id',how='left',validate='m:1')

In [7]:
# 서비스 상세의 독립 집계: 줄 단가를 카탈로그와 검증하고 오류가 있는 그룹은 복구 근거에서 제외
catalog=load('service_catalog',['service_version_id','service_code']).with_columns(
    pl.col('valid_from').str.replace_all('/','-').str.to_date('%Y-%m-%d',strict=False),
    pl.col('valid_to').str.replace_all('/','-').str.to_date('%Y-%m-%d',strict=False),
    pl.col('unit_price').cast(pl.Int64,strict=False))
cat_counts,cat=unique_reference(catalog,'service_version_id')
services=load('encounter_services',['encounter_service_id','encounter_id','service_version_id']).with_columns([
    pl.col(c).cast(pl.Int64,strict=False) for c in ['line_no','quantity','unit_price','line_amount']])
services=services.join(cat.select('service_version_id',pl.col('unit_price').alias('catalog_unit_price'),'valid_from','valid_to'),on='service_version_id',how='left',validate='m:1')
# 날짜는 진료별로 교차검증 (없는 날짜는 근거 불충분)
enc_dates=base.filter(pl.col('encounter_id').is_not_null()&(pl.len().over('encounter_id')==1)).select('encounter_id',pl.col('started_at').dt.date().alias('encounter_date'))
services=services.join(enc_dates,on='encounter_id',how='left',validate='m:1').with_columns(
    (pl.col('encounter_service_id').is_not_null()
     &(pl.len().over('encounter_service_id')==1)
     &(pl.len().over(['encounter_id','line_no'])==1)
     &(pl.col('line_no')>0)&(pl.col('quantity')>0)&(pl.col('unit_price')>0)
     &(pl.col('line_amount')==pl.col('quantity')*pl.col('unit_price'))
     &(pl.col('unit_price')==pl.col('catalog_unit_price'))
     &pl.col('encounter_date').is_between(pl.col('valid_from'),pl.col('valid_to'),closed='both')
     &pl.col('source_system').is_in(['LEGACY_A','LEGACY_B'])).fill_null(False).alias('service_evidence_valid'))
service_summary=services.filter(pl.col('encounter_id').is_not_null()).group_by('encounter_id').agg(
    pl.col('line_amount').sum().alias('service_gross'),
    (~pl.col('service_evidence_valid')).sum().alias('service_invalid_rows'),pl.len().alias('service_rows'))

In [8]:
# 결제·환불의 독립 집계: 거래 ID 충돌, 양수 금액, 원결제·시간·누적 환불 검증
payments=map_codes(load('payments',['payment_id','encounter_id','original_payment_id']),'transaction_type','transaction_type').unique().with_columns(
    pl.col('amount').cast(pl.Int64,strict=False),parse_dt('transacted_at'))
payments=payments.with_columns(
    (pl.col('payment_id').is_not_null()&(pl.len().over('payment_id')==1)
     &pl.col('encounter_id').is_not_null()&(pl.col('amount')>0)
     &pl.col('transacted_at').is_not_null()&pl.col('transaction_type').is_in(['PAYMENT','REFUND'])
     &pl.col('source_system').is_in(['LEGACY_A','LEGACY_B'])).fill_null(False).alias('basic_valid'))
original=payments.filter(pl.col('basic_valid')&(pl.col('transaction_type')=='PAYMENT')&pl.col('original_payment_id').is_null()).select(
    pl.col('payment_id').alias('original_payment_id'),pl.col('encounter_id').alias('original_encounter_id'),pl.col('amount').alias('original_amount'),pl.col('transacted_at').alias('original_at'))
refund_totals=payments.filter(pl.col('transaction_type')=='REFUND').group_by('original_payment_id').agg(pl.col('amount').sum().alias('total_refunded'),pl.col('amount').null_count().alias('refund_parse_failures'))
payments=payments.join(original,on='original_payment_id',how='left',validate='m:1').join(refund_totals,on='original_payment_id',how='left',validate='m:1').with_columns(
    (pl.col('basic_valid')&(
        ((pl.col('transaction_type')=='PAYMENT')&pl.col('original_payment_id').is_null())|
        ((pl.col('transaction_type')=='REFUND')&(pl.col('encounter_id')==pl.col('original_encounter_id'))
         &(pl.col('transacted_at')>=pl.col('original_at'))&(pl.col('total_refunded')<=pl.col('original_amount'))
         &(pl.col('refund_parse_failures')==0)))).fill_null(False).alias('payment_evidence_valid'))
payment_summary=payments.filter(pl.col('encounter_id').is_not_null()).group_by('encounter_id').agg(
    pl.when(pl.col('transaction_type')=='PAYMENT').then(pl.col('amount')).otherwise(0).sum().alias('payment_gross'),
    pl.when(pl.col('transaction_type')=='REFUND').then(pl.col('amount')).otherwise(0).sum().alias('payment_refund'),
    (~pl.col('payment_evidence_valid')).sum().alias('payment_invalid_rows'),
    (pl.col('transaction_type')=='PAYMENT').sum().alias('payment_rows'))
work=base.join(service_summary,on='encounter_id',how='left',validate='m:1').join(payment_summary,on='encounter_id',how='left',validate='m:1')
work=work.with_columns((pl.col('payment_gross')-pl.col('payment_refund')).alias('payment_net'))

In [9]:
# 독립 근거의 일치를 이용한 복구. 원본 값은 그대로 남김.
strong=(pl.col('service_invalid_rows')==0)&(pl.col('payment_invalid_rows')==0)&(pl.col('payment_rows')>0)&(pl.col('service_gross')==pl.col('payment_gross'))
work=work.with_columns(strong.fill_null(False).alias('strong_gross_evidence'))
work=work.with_columns([
    pl.col(c).alias('before_'+c) for c in MONEY
]).with_columns(
    pl.when(pl.col('strong_gross_evidence')).then(pl.col('service_gross')).otherwise(pl.col('gross_charge')).alias('gross_charge'))
refund_safe=(pl.col('payment_invalid_rows')==0)&(pl.col('payment_rows')>0)&(pl.col('gross_charge')==pl.col('payment_gross'))&(pl.col('net_charge')==pl.col('payment_net'))&((pl.col('gross_charge')-pl.col('net_charge'))==pl.col('payment_refund'))
work=work.with_columns(pl.when(refund_safe).then(pl.col('payment_refund')).otherwise(pl.col('authorized_refund')).alias('authorized_refund'))
net_safe=(pl.col('payment_invalid_rows')==0)&(pl.col('payment_rows')>0)&(pl.col('gross_charge')==pl.col('payment_gross'))&(pl.col('authorized_refund')==pl.col('payment_refund'))
work=work.with_columns(pl.when(net_safe).then(pl.col('gross_charge')-pl.col('authorized_refund')).otherwise(pl.col('net_charge')).alias('net_charge'))
audit_parts=[]
for c,reason in [('gross_charge','valid service sum = valid payment sum'),('authorized_refund','valid refund sum = corroborated gross - original net'),('net_charge','gross - approved refund = valid payment - refund')]:
    audit_parts.append(work.filter(~pl.col(c).eq_missing(pl.col('before_'+c))).select('encounter_id','source_file','source_record',pl.lit(c).alias('column'),pl.col('raw_'+c).alias('original_value'),pl.col(c).cast(pl.String).alias('clean_value'),pl.lit(reason).alias('evidence')))
# 표기 표준화 이력: 중복 제거 전 모든 원본에 대해 기록
for c in ['encounter_id','checkin_id','started_at','ended_at','source_system']:
    audit_parts.append(text.filter(~pl.col(c).eq_missing(pl.col('raw_'+c))).select('encounter_id','source_file','source_record',pl.lit(c).alias('column'),pl.col('raw_'+c).alias('original_value'),pl.col(c).alias('clean_value'),pl.lit('B15 deterministic notation normalization').alias('evidence')))
repair_log=pl.concat(audit_parts,how='vertical_relaxed')

In [10]:
rules={}
for c in COLS:rules['missing_or_unparseable_'+c]=pl.col(c).is_null()
rules.update({
    'invalid_encounter_id':~pl.col('encounter_id').str.contains(r'^ENC_\d{9}$'),
    'invalid_checkin_id':~pl.col('checkin_id').str.contains(r'^CHK_\d{9}$'),
    'duplicate_encounter_id':pl.col('encounter_id').is_not_null()&(pl.len().over('encounter_id')>1),
    'duplicate_checkin_id':pl.col('checkin_id').is_not_null()&(pl.len().over('checkin_id')>1),
    'invalid_source_system':~pl.col('source_system').is_in(['LEGACY_A','LEGACY_B']),
    'unknown_checkin':pl.col('checkin_id').is_not_null()&pl.col('checkin_reference_count').is_null(),
    'ambiguous_checkin':pl.col('checkin_reference_count')>1,
    'missing_checkin_outcome':(pl.col('checkin_reference_count')==1)&pl.col('outcome').is_null(),
    'source_system_mismatch':pl.col('source_system')!=pl.col('checkin_source_system'),
    'not_seen_checkin':pl.col('outcome').is_not_null()&(pl.col('outcome')!='SEEN'),
    'unverified_checkin_time':(pl.col('checkin_reference_count')==1)&(pl.col('arrival_at').is_null()|pl.col('checked_in_at').is_null()|pl.col('session_end').is_null()),
    'checkin_arrival_order':pl.col('arrival_at')>pl.col('checked_in_at'),
    'starts_before_checkin':pl.col('started_at')<pl.col('checked_in_at'),
    'invalid_encounter_time_order':pl.col('started_at')>=pl.col('ended_at'),
    'ends_after_session':pl.col('ended_at')>pl.col('session_end'),
    'negative_or_overflow_amount':pl.any_horizontal([(pl.col(c)<0)|(pl.col(c)>2147483647) for c in MONEY]),
    'refund_exceeds_gross':pl.col('authorized_refund')>pl.col('gross_charge'),
    'net_mismatch':pl.col('net_charge')!=pl.col('gross_charge')-pl.col('authorized_refund'),
    'service_evidence_unavailable':pl.col('service_rows').is_null()|(pl.col('service_invalid_rows')>0),
    'payment_evidence_unavailable':pl.col('payment_rows').is_null()|(pl.col('payment_rows')==0)|(pl.col('payment_invalid_rows')>0),
    'service_gross_mismatch':(pl.col('service_invalid_rows')==0)&(pl.col('gross_charge')!=pl.col('service_gross')),
    'payment_net_mismatch':(pl.col('payment_invalid_rows')==0)&(pl.col('net_charge')!=pl.col('payment_net')),
    'approved_actual_refund_mismatch':(pl.col('payment_invalid_rows')==0)&(pl.col('authorized_refund')!=pl.col('payment_refund')),
})
checked=work.with_columns([e.fill_null(False).alias(k) for k,e in rules.items()]).with_columns(
    pl.concat_list([pl.when(pl.col(k)).then(pl.lit(k)).otherwise(None) for k in rules]).list.drop_nulls().alias('issue_reasons'))
quality_report=pl.DataFrame({'rule':list(rules),'rows':[checked[k].sum() for k in rules]}).sort('rows',descending=True)

## 6. 클리닝 결과
현재 명세서 규칙과 교차검증을 모두 통과한 행만 clean에 포함합니다. 참조 데이터의 오류·누락으로 검증이 불가능한 행도 review에 남깁니다. 이 분류는 관련 테이블의 최종 정제에 따라 바뀔 수 있습니다.

In [11]:
encounters_clean=checked.filter(pl.col('issue_reasons').list.len()==0).select(COLS+['source_file','source_record'])
encounters_review=checked.filter(pl.col('issue_reasons').list.len()>0)
assert raw.height==removed_duplicate_count+encounters_clean.height+encounters_review.height
assert encounters_clean['encounter_id'].n_unique()==encounters_clean.height
assert encounters_clean['checkin_id'].n_unique()==encounters_clean.height
assert encounters_clean.select(pl.col(COLS).null_count()).row(0)==(0,)*len(COLS)
summary=pl.DataFrame({'stage':['raw','duplicate_removed','clean','review'],'rows':[raw.height,removed_duplicate_count,encounters_clean.height,encounters_review.height]})
repair_summary=repair_log.group_by('column','evidence').len().sort('column')
seen_without_encounter=chk.filter(pl.col('outcome')=='SEEN').join(checked.select('checkin_id').unique(),on='checkin_id',how='anti')
display(summary)
display(quality_report)
display(repair_summary)
display(repair_log.head(10))
display(encounters_clean.head(5))
display(encounters_review.select(COLS+['issue_reasons']).head(10))
print('SEEN checkins without linked encounter:',seen_without_encounter.height)
print('동일 내용 중복 그룹 행:',duplicate_rows.height,'/ 제거 행:',removed_duplicate_count)

shape: (4, 2)
┌───────────────────┬─────────┐
│ stage             ┆ rows    │
│ ---               ┆ ---     │
│ str               ┆ i64     │
╞═══════════════════╪═════════╡
│ raw               ┆ 1004999 │
│ duplicate_removed ┆ 4999    │
│ clean             ┆ 740628  │
│ review            ┆ 259372  │
└───────────────────┴─────────┘
shape: (31, 2)
┌─────────────────────────────────┬────────┐
│ rule                            ┆ rows   │
│ ---                             ┆ ---    │
│ str                             ┆ i64    │
╞═════════════════════════════════╪════════╡
│ service_evidence_unavailable    ┆ 159482 │
│ payment_evidence_unavailable    ┆ 46001  │
│ service_gross_mismatch          ┆ 28633  │
│ payment_net_mismatch            ┆ 21504  │
│ missing_or_unparseable_checkin… ┆ 10004  │
│ …                               ┆ …      │
│ source_system_mismatch          ┆ 0      │
│ checkin_arrival_order           ┆ 0      │
│ starts_before_checkin           ┆ 0      │
│ ends_after_session 

## 7. 정제 결과 파일 저장
최종 정제 데이터는 data/output/encounters.csv에 업무 컬럼만 저장합니다. Polars는 pandas의 인덱스와 같은 별도 번호 컬럼을 생성하지 않습니다. 추적용 source_record도 최종 파일에서 제외합니다.

UTF-8 BOM(utf-8-sig), 날짜 YYYY-MM-DD, 일시 YYYY-MM-DD HH:MM:SS를 사용하고 실제 결측값은 빈칸으로 저장합니다. 검토 대상·변경 이력·검증 보고서는 data/output/reports/encounters/에 같은 기준으로 저장합니다. 원본 값을 기록한 검토 자료에는 오염된 문자열이 그대로 남을 수 있습니다.

각 노트북은 기존 정제 규칙으로 독립 실행하며, 다른 노트북의 복구 결과가 자동 연결되는 것은 아닙니다. MySQL 적재는 수행하지 않습니다.

최종 정제본에서 제외된 검토 대상과 실제 중복 제거 행은 data/output/encounters_dq.csv에 합쳐 저장합니다. dq_type은 REVIEW 또는 DUPLICATE_EXCLUDED, dq_reason은 검토 사유입니다. 업무 컬럼에는 수정 전 원본 값을 넣고 파일명과 레코드 순번을 남깁니다. 중복 그룹의 대표 행은 중복 제외 목록에 넣지 않습니다. 제외 행이 없더라도 헤더가 있는 빈 파일을 생성합니다.

In [12]:
from pathlib import Path
import json
FINAL_DIR = ROOT / 'data/output'
FINAL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = FINAL_DIR / 'reports' / 'encounters'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
def export_csv(frame, filename):
    # CSV에 중첩 자료형을 직접 저장할 수 없어 사유 목록을 구분 문자열로 보존
    out = frame.with_columns([pl.col(c).list.join('|').alias(c)
        for c, dtype in frame.schema.items() if isinstance(dtype, pl.List)])
    destination = FINAL_DIR / filename if filename in ['encounters.csv', 'encounters_dq.csv'] else OUTPUT_DIR / filename
    out.write_csv(destination, include_bom=True, null_value='',
        datetime_format='%Y-%m-%d %H:%M:%S', date_format='%Y-%m-%d')
    return {'file': filename, 'rows': frame.height, 'columns': frame.width}
exported = []
business_columns = COLS
exported.append(export_csv(encounters_clean.select(business_columns), 'encounters.csv'))
# DQ에는 검토 행과 실제 제거된 중복 행을 원본 값으로 기록
provenance = ['source_file', 'source_record']
retained_locations = base.select(provenance)
excluded_locations = text.select(provenance).join(retained_locations, on=provenance, how='anti')
review_reasons = encounters_review.select(provenance + ['issue_reasons']).with_columns(
    pl.col('issue_reasons').list.join('|').alias('dq_reason')).drop('issue_reasons')
dq_review = raw.join(review_reasons, on=provenance, how='inner', validate='1:1').with_columns(
    pl.lit('REVIEW').alias('dq_type'))
dq_duplicates = raw.join(excluded_locations, on=provenance, how='inner', validate='1:1').with_columns(
    pl.lit('DUPLICATE_EXCLUDED').alias('dq_type'),
    pl.lit('identical_normalized_content').alias('dq_reason'))
dq_columns = business_columns + provenance + ['dq_type', 'dq_reason']
encounters_dq = pl.concat([dq_review.select(dq_columns), dq_duplicates.select(dq_columns)])
expected_excluded = summary.filter(pl.col('stage') == 'duplicate_removed')['rows'][0]
assert dq_duplicates.height == expected_excluded
assert dq_review.height == encounters_review.height
assert raw.height == encounters_clean.height + encounters_dq.height
assert encounters_dq.select(provenance).unique().height == encounters_dq.height
assert encounters_dq.select(provenance).join(encounters_clean.select(provenance), on=provenance, how='inner').is_empty()
exported.append(export_csv(encounters_dq, 'encounters_dq.csv'))
loaded_dq = pl.read_csv(FINAL_DIR / 'encounters_dq.csv', infer_schema=False)
assert loaded_dq.height == encounters_dq.height and loaded_dq.columns == dq_columns

exported.append(export_csv(encounters_review, 'review.csv'))
for variable in ['summary', 'quality_report', 'raw_report', 'repair_log', 'repair_summary',
                 'reconciliation', 'reconciliation_report', 'reference_report',
                 'duplicate_rows', 'seen_without_encounter']:
    value = globals().get(variable)
    if isinstance(value, pl.DataFrame):
        exported.append(export_csv(value, variable + '.csv'))
# 전송 결과의 행 수 및 스키마 확인 (자료형은 CSV 밖 manifest에 기록)
loaded = pl.read_csv(FINAL_DIR / 'encounters.csv', infer_schema=False)
assert loaded.height == encounters_clean.height and loaded.columns == business_columns
manifest = {'table': 'encounters', 'polars_version': pl.__version__,
    'clean_schema': {c: str(dtype) for c, dtype in encounters_clean.select(business_columns).schema.items()},
    'null_token': '', 'list_separator': '|', 'encoding': 'utf-8-sig',
    'execution_mode': 'independent notebooks using original reference data', 'files': exported}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('Final CSV:', FINAL_DIR / 'encounters.csv')
print('Reports:', OUTPUT_DIR)
display(pl.DataFrame(exported))


Final CSV: C:\workspace\medallion_venv\data\output\encounters.csv
Reports: C:\workspace\medallion_venv\data\output\reports\encounters
shape: (10, 3)
┌────────────────────────────┬────────┬─────────┐
│ file                       ┆ rows   ┆ columns │
│ ---                        ┆ ---    ┆ ---     │
│ str                        ┆ i64    ┆ i64     │
╞════════════════════════════╪════════╪═════════╡
│ encounters.csv             ┆ 740628 ┆ 8       │
│ encounters_dq.csv          ┆ 264371 ┆ 12      │
│ review.csv                 ┆ 259372 ┆ 68      │
│ summary.csv                ┆ 4      ┆ 2       │
│ quality_report.csv         ┆ 31     ┆ 2       │
│ raw_report.csv             ┆ 18     ┆ 2       │
│ repair_log.csv             ┆ 20004  ┆ 7       │
│ repair_summary.csv         ┆ 2      ┆ 3       │
│ duplicate_rows.csv         ┆ 9998   ┆ 18      │
│ seen_without_encounter.csv ┆ 20004  ┆ 6       │
└────────────────────────────┴────────┴─────────┘
